In [ ]:
import bs4
import json
import urllib
from typing import Union
import pathlib as pl
from icecream import ic
import zotero_to_obsidian_note_listener as zol

### The Zotero Note HTML to Obsidian Markdown Converter

The real code is in the webhook listener, but it's nice to work in a notebook, sometimes.

### Converter Test

In [29]:
test_json_input_file = pl.Path(r"C:\Users\scott\tmp\zotero_item_dat.json")
data = json.loads(test_json_input_file.read_text(encoding="utf-8"))
input_html = data[0]['notes'][0]  # first zotero item in json, 1st note in that item

In [30]:
html_out_file = pl.Path(r"C:\Users\scott\tmp")  / "tmp.html"
html_out_file.write_text(input_html, encoding="utf-8") # for per-file comparison of json extraction

#output_markdown = z2o.zotero_note_html_to_md(input_html)
output_markdown = zotero_note_html_to_md(input_html)
markdown_out_file = pl.Path(r"C:\Users\scott\OneDrive\share\ref\obsidian\Obsidian Share Vault\Scratch Space\zot_note_to_obs.md")
markdown_out_file.write_text(output_markdown, encoding="utf-8")
ic(test_json_input_file, html_out_file, markdown_out_file);

# direct comparison
outdir = pl.Path(r"C:\Users\scott\tmp")
(outdir / "note_convert_test_JSON_html.html").write_text(input_html,encoding='utf-8')
(outdir / "note_convert_test_markdown.md").write_text(output_markdown,encoding='utf-8')

ic| test_json_input_file: WindowsPath('C:/Users/scott/tmp/zotero_item_dat.json')
    html_out_file: WindowsPath('C:/Users/scott/tmp/tmp.html')
    markdown_out_file: WindowsPath('C:/Users/scott/OneDrive/share/ref/obsidian/Obsidian Share Vault/Scratch Space/zot_note_to_obs.md')


996

In [31]:
input_html

'<div data-citation-items="%5B%7B%22uris%22%3A%5B%22http%3A%2F%2Fzotero.org%2Fusers%2F60638%2Fitems%2FB7TNABNU%22%5D%2C%22itemData%22%3A%7B%22id%22%3A%22http%3A%2F%2Fzotero.org%2Fusers%2F60638%2Fitems%2FB7TNABNU%22%2C%22type%22%3A%22webpage%22%2C%22abstract%22%3A%22Offered%20by%20University%20of%20California%2C%20Davis.%20As%20data%20collection%20has%20increased%20exponentially%2C%20so%20has%20the%20need%20for%20people%20skilled%20at%20using%20and%20...%20Enroll%20for%20free.%22%2C%22container-title%22%3A%22Coursera%22%2C%22language%22%3A%22en%22%2C%22note%22%3A%22Citation%20Key%3A%20Lawrence22dataScienceSQLcourse%22%2C%22title%22%3A%22SQL%20for%20Data%20Science%22%2C%22URL%22%3A%22https%3A%2F%2Fwww.coursera.org%2Flearn%2Fsql-for-data-science%22%2C%22author%22%3A%5B%7B%22family%22%3A%22Lawrence%22%2C%22given%22%3A%22Sadie%22%2C%22suffix%22%3A%22St.%22%7D%5D%2C%22accessed%22%3A%7B%22date-parts%22%3A%5B%5B%222025%22%2C1%2C13%5D%5D%7D%2C%22issued%22%3A%7B%22date-parts%22%3A%5B%5B%222022%2

In [32]:
output_markdown

'MongoDB25noSQLVsSQLdatabases\n\n## Points\n\n- Recommended reading in SQL course: ([Lawrence, 2022](zotero://select/library/items/B7TNABNU))\n- clear graph of a set of foreign keys\n\n## SQL\n\n- big list of SQL variants\n- scales “vertically”: make a single server bigger\n- ACID stingent security complant (always)\n- SQL use cases\n\n\nACID required by regulations\n\n\ntransactional\n\n\nenterprise resource planning e.g. supply chain, human resources,…\n\n## NoSQL\n\n- not always SQL (can do SQL too)\n- scaled “horizontally”: I think this means can expand by adding a new compute node\n- use when data changes fast, must be scalable, and when it’s non-structured\n- usually doesn’t meet stringent ACID standards\n\n\nSQL often does\n\n\nsome NoSQL does e.g. Mongo’s\n- big list of NoSQL types\n\n\nDocument\n\n\nKey-value\n\n\nColumn-family stores\n\n\nGraph\n- NoSQL use cases\n\n\ntransactional (can just do internal SQL-type tables), or when store unstructured\n\n\ndocument and digital as

### Tinker with BeautifulSoup

In [33]:
# Parse the HTML
soup = bs4.BeautifulSoup(input_html, 'html.parser')  # or use 'lxml' for better handling

In [34]:
# Find the div with citation data
citation_div = soup.find('div', attrs={'data-citation-items': True})

# Get the JSON data from the attribute
citation_json = citation_div['data-citation-items']

# The JSON is URL-encoded, so you may need to decode it
import urllib.parse
decoded_json = urllib.parse.unquote(citation_json)

# Parse the JSON data
citation_data = json.loads(decoded_json)


In [35]:
print(decoded_json)

[{"uris":["http://zotero.org/users/60638/items/B7TNABNU"],"itemData":{"id":"http://zotero.org/users/60638/items/B7TNABNU","type":"webpage","abstract":"Offered by University of California, Davis. As data collection has increased exponentially, so has the need for people skilled at using and ... Enroll for free.","container-title":"Coursera","language":"en","note":"Citation Key: Lawrence22dataScienceSQLcourse","title":"SQL for Data Science","URL":"https://www.coursera.org/learn/sql-for-data-science","author":[{"family":"Lawrence","given":"Sadie","suffix":"St."}],"accessed":{"date-parts":[["2025",1,13]]},"issued":{"date-parts":[["2022"]]},"citation-key":"Lawrence22dataScienceSQLcourse"}}]


In [36]:
citation_data

[{'uris': ['http://zotero.org/users/60638/items/B7TNABNU'],
  'itemData': {'id': 'http://zotero.org/users/60638/items/B7TNABNU',
   'type': 'webpage',
   'abstract': 'Offered by University of California, Davis. As data collection has increased exponentially, so has the need for people skilled at using and ... Enroll for free.',
   'container-title': 'Coursera',
   'language': 'en',
   'note': 'Citation Key: Lawrence22dataScienceSQLcourse',
   'title': 'SQL for Data Science',
   'URL': 'https://www.coursera.org/learn/sql-for-data-science',
   'author': [{'family': 'Lawrence', 'given': 'Sadie', 'suffix': 'St.'}],
   'accessed': {'date-parts': [['2025', 1, 13]]},
   'issued': {'date-parts': [['2022']]},
   'citation-key': 'Lawrence22dataScienceSQLcourse'}}]

In [37]:
# Get all text within the div
text_content = citation_div.get_text(' ', strip=True)

# Find all citation spans
citations = citation_div.find_all('span', class_='citation-item')
citation_texts = [citation.text for citation in citations]


In [38]:
citation_texts

['Lawrence, 2022']

In [39]:
print(soup.prettify())

<div data-citation-items="%5B%7B%22uris%22%3A%5B%22http%3A%2F%2Fzotero.org%2Fusers%2F60638%2Fitems%2FB7TNABNU%22%5D%2C%22itemData%22%3A%7B%22id%22%3A%22http%3A%2F%2Fzotero.org%2Fusers%2F60638%2Fitems%2FB7TNABNU%22%2C%22type%22%3A%22webpage%22%2C%22abstract%22%3A%22Offered%20by%20University%20of%20California%2C%20Davis.%20As%20data%20collection%20has%20increased%20exponentially%2C%20so%20has%20the%20need%20for%20people%20skilled%20at%20using%20and%20...%20Enroll%20for%20free.%22%2C%22container-title%22%3A%22Coursera%22%2C%22language%22%3A%22en%22%2C%22note%22%3A%22Citation%20Key%3A%20Lawrence22dataScienceSQLcourse%22%2C%22title%22%3A%22SQL%20for%20Data%20Science%22%2C%22URL%22%3A%22https%3A%2F%2Fwww.coursera.org%2Flearn%2Fsql-for-data-science%22%2C%22author%22%3A%5B%7B%22family%22%3A%22Lawrence%22%2C%22given%22%3A%22Sadie%22%2C%22suffix%22%3A%22St.%22%7D%5D%2C%22accessed%22%3A%7B%22date-parts%22%3A%5B%5B%222025%22%2C1%2C13%5D%5D%7D%2C%22issued%22%3A%7B%22date-parts%22%3A%5B%5B%222022%22